In [1]:
import snapatac2 as snap
import numpy as np
import pandas as pd
# Set the environment variable to the desired cache path
import os
import scanpy as sc

In [11]:
neuron_peaks = sc.read_h5ad("/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Snapatac2/Neuron/peak_mat.h5ad")
anno = pd.read_csv("/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scanpy/Neuron/neuron_tri_lineage.csv", index_col=0)
# 将 dIPC, dIPC_switching, dIPC_switched 统一替换为 IPC
anno['celltype'] = anno['celltype'].replace(['dIPC', 'dIPC_switching', 'dIPC_switched'], 'IPC')
# neuron_peaks.obs["leiden"] = anno.loc[neuron_peaks.obs_names, "leiden"]
# neuron_peaks.obs['leiden'] = neuron_peaks.obs['leiden'].astype(str)
neuron_peaks.obs["is_ExN_lineage"] = anno.loc[neuron_peaks.obs_names, "is_ExN_lineage"]
neuron_peaks.obs['is_ExN_lineage'] = neuron_peaks.obs['is_ExN_lineage'].astype(str)
# neuron_peaks.obs['leiden'] = neuron_peaks.obs['leiden'].astype(str)
# neuron_peaks.obs['leiden'] = neuron_peaks.obs['leiden'].astype(str)


In [12]:
neuron_peaks.obs['is_ExN_lineage']

AAACAGCCATCGTTCT-1-0    1
AAACCGAAGACCATAC-1-0    1
AAACCGAAGGTCCACA-1-0    1
AAACCGAAGTCATGCG-1-0    1
AAACCGGCAAATTGCT-1-0    1
                       ..
TTTGTGAAGCGCCTAA-1-7    0
TTTGTGGCAAGATTCT-1-7    0
TTTGTGGCAAGTCGCT-1-7    0
TTTGTGGCACGTAATT-1-7    0
TTTGTTGGTCACAAAT-1-7    0
Name: is_ExN_lineage, Length: 10248, dtype: object

In [13]:
neuron_peaks.obs['lineage_celltype'] = (
    neuron_peaks.obs['celltype'].astype(str)
    + "_" 
    + neuron_peaks.obs['is_ExN_lineage'].astype(str)
).astype("category")

In [15]:
import pandas as pd
import scanpy as sc
import snapatac2 as snap
import os

# 1. 定义路径和读取数据
h5ad_path = "/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Snapatac2/Neuron/peak_mat.h5ad"
csv_path = "/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scanpy/Neuron/neuron_tri_lineage.csv"
base_out_dir = "/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig4"

print("Loading data...")
neuron_peaks = sc.read_h5ad(h5ad_path)
anno = pd.read_csv(csv_path, index_col=0)

# 2. 预处理：统一修改 celltype 和同步 metadata
# 修改 dIPC -> IPC
anno['celltype'] = anno['celltype'].replace(['dIPC', 'dIPC_switching', 'dIPC_switched'], 'IPC')

# 确保 anno 的顺序和 neuron_peaks 的 obs_names 一致，并更新到 neuron_peaks.obs
# 注意：这里假设 anno 包含了 lineage 列 (is_ExN_lineage 等)
neuron_peaks.obs = anno.loc[neuron_peaks.obs_names].copy()

# 确保 celltype 是字符串或分类类型，避免后续报错
neuron_peaks.obs['celltype'] = neuron_peaks.obs['celltype'].astype(str)

# 3. 定义需要处理的三个谱系列名和对应的输出文件夹名
# 格式： "CSV中的列名": "输出文件夹名"
lineage_map = {
    'is_ExN_lineage': 'ExN',
    'is_InN_lineage': 'InN',
    'is_Oligo_lineage': 'Oligo'
}

# 4. 批量循环处理
for col_name, lineage_name in lineage_map.items():
    print(f"Processing lineage: {lineage_name} (Column: {col_name})...")
    
    # 步骤 A: 筛选属于当前谱系的细胞
    # 假设 CSV 中 1 表示属于该谱系，0 表示不属于
    mask = neuron_peaks.obs[col_name] == 1
    
    # 如果该谱系没有细胞，跳过
    if mask.sum() == 0:
        print(f"  No cells found for {lineage_name}, skipping.")
        continue
        
    # 创建该谱系的子集 (View)
    subset_adata = neuron_peaks[mask].copy()
    
    print(f"  Selected {subset_adata.n_obs} cells.")
    print(f"  Cell types in this lineage: {subset_adata.obs['celltype'].unique()}")

    # 步骤 B: 定义该谱系的输出路径 (例如: .../bigwig4/ExN/)
    out_dir = os.path.join(base_out_dir, lineage_name)
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    
    # 步骤 C: 生成 BigWig
    # 直接使用 clean 过的 'celltype' 列作为 groupby
    # 不需要拼接字符串，因为我们已经通过文件夹区分了谱系
    try:
        snap.ex.export_coverage(
            subset_adata, 
            groupby="celltype", 
            out_dir=out_dir, 
            output_format="bigwig",
            suffix=".bw" # 可选，明确后缀
        )
        print(f"  Export finished for {lineage_name} -> {out_dir}")
    except Exception as e:
        print(f"  Error exporting {lineage_name}: {e}")

print("All Done.")

Loading data...
Processing lineage: ExN (Column: is_ExN_lineage)...


2026-03-03 17:23:17 - INFO - Exporting fragments...


  Selected 3913 cells.
  Cell types in this lineage: ['NPC' 'IPC' 'ExN_naive']


2026-03-03 17:24:28 - INFO - Creating coverage files...


  Export finished for ExN -> /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig4/ExN
Processing lineage: InN (Column: is_InN_lineage)...


...

  Export finished for InN -> /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig4/InN
Processing lineage: Oligo (Column: is_Oligo_lineage)...


2026-03-03 17:28:17 - INFO - Exporting fragments...


  Selected 4649 cells.
  Cell types in this lineage: ['IPC' 'OPC' 'NPC']


2026-03-03 17:29:47 - INFO - Creating coverage files...


  Export finished for Oligo -> /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig4/Oligo
All Done.


In [14]:
# snap.ex.export_coverage(neuron_peaks, groupby="celltype", out_dir="/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig", output_format="bigwig")
# snap.ex.export_coverage(neuron_peaks, groupby="leiden", out_dir="/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig2", output_format="bigwig")
# snap.ex.export_coverage(neuron_peaks, groupby="time_celltype", out_dir="/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig3", output_format="bigwig")
snap.ex.export_coverage(neuron_peaks, groupby="lineage_celltype", out_dir="/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig4", output_format="bigwig")

2026-03-03 16:43:57 - INFO - Exporting fragments...
2026-03-03 16:46:41 - INFO - Creating coverage files...


{'NPC_0': '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig4/NPC_0.bw',
 'dIPC_1': '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig4/dIPC_1.bw',
 'NPC_1': '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig4/NPC_1.bw',
 'ExN_naive_0': '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig4/ExN_naive_0.bw',
 'InN_naive_0': '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig4/InN_naive_0.bw',
 'OPC_0': '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig4/OPC_0.bw',
 'ExN_naive_1': '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig4/ExN_naive_1.bw',
 'dIPC_switched_0': '/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/bigwig4/dIPC_switched_0.bw',
 'dIPC_switching_1': '/cluster2/huanglab/jiamao/Projec

In [ ]:
peak_df = neuron_peaks.var.index.to_series().str.extract(r'(chr[\w]+):(\d+)-(\d+)')
peak_df.columns = ['chrom', 'start', 'end']
peak_df.to_csv("/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Neuron/outs/consensus_peaks.bed", sep="\t", header=False, index=False)

In [ ]:
print(neuron_peaks.var.head())

Empty DataFrame
Columns: []
Index: [chr1:817093-817594, chr1:817859-818360, chr1:827302-827803, chr1:849801-850302, chr1:858640-859141]
